In [1]:
"""
Pre2: Test multiple regression configurations
Compare: half-year vs annual, with/without 2025, with/without dummy
"""
import glob, os
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

DATA_DIR = "./data"

# ============================================================
# 1. Load ERCOT data (2018-2025, 8 files)
# ============================================================
dfs = []
for f in sorted(glob.glob(os.path.join(DATA_DIR, "Native_Load_*.xlsx"))):
    df = pd.read_excel(f)
    df.columns = df.columns.str.strip().str.replace(" ", "")
    raw = df['HourEnding'].astype(str)
    ending_hour = raw.str.extract(r'(\d{1,2}):\d{2}')[0].astype(float)
    df['hour'] = (ending_hour - 1).astype(int)
    raw_fixed = raw.str.replace(' 24:00', ' 00:00', regex=False)
    raw_fixed = raw_fixed.str.replace('24:00:00', '00:00:00', regex=False)
    df['HourEnding'] = pd.to_datetime(raw_fixed, errors='coerce')
    df = df.dropna(subset=['HourEnding'])
    df['year'] = df['HourEnding'].dt.year
    df['month'] = df['HourEnding'].dt.month
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(df_all)}, Years: {df_all['year'].min()}-{df_all['year'].max()}")

# ============================================================
# 2. DFW DC inventory (extended to 2025)
# ============================================================
dfw_dc = pd.DataFrame({
    'period': ['H1 2018','H2 2018','H1 2019','H2 2019',
               'H1 2020','H2 2020','H1 2021','H2 2021',
               'H1 2022','H2 2022','H1 2023','H2 2023',
               'H1 2024','H2 2024','H1 2025','H2 2025'],
    'dc_mw':  [273.1, 306.1, 321.4, 321.4,
               347.4, 360.9, 367.3, 369.3,
               375.8, 392.3, 499.5, 565.4,
               591.0, 591.0, 869.5, 1067.3]
})

# ============================================================
# 3. NCENT semi-annual average
# ============================================================
df_all['half'] = df_all['month'].apply(lambda m: 'H1' if m <= 6 else 'H2')
df_all['period'] = df_all['half'] + ' ' + df_all['year'].astype(str)

ncent_semi = df_all.groupby('period')['NCENT'].mean().round(2).reset_index()
ncent_semi.columns = ['period', 'ncent_mw']

# ============================================================
# 4. Merge and build features
# ============================================================
reg = dfw_dc.merge(ncent_semi, on='period')
reg['year'] = reg['period'].str.extract(r'(\d{4})').astype(int)
reg['summer'] = reg['period'].str.startswith('H2').astype(int)
reg['time'] = range(len(reg))
reg['post2022'] = (reg['year'] >= 2022).astype(int)

print(f"\nFull dataset ({len(reg)} rows):")
print(reg.to_string(index=False))

# ============================================================
# 5. Run 4 configurations
# ============================================================
print("\n" + "="*50)
print("CONFIGURATION TESTS")
print("="*50)

def run_model(label, data, y_col, x_cols):
    y = data[y_col].values
    X = data[x_cols].values
    m = LinearRegression().fit(X, y)
    r2 = r2_score(y, m.predict(X))
    coefs = dict(zip(x_cols, m.coef_.round(4)))
    print(f"\n{label}")
    print(f"  n={len(data)}, R²={r2:.4f}")
    print(f"  Coefficients: {coefs}")
    return m

# Test A: 2018-2025 half-year (16 points)
print("\n--- A: Half-year 2018-2025 ---")
run_model("A0: Time+Season", reg, 'ncent_mw', ['time','summer'])
run_model("A1: Time+Season+DC", reg, 'ncent_mw', ['time','summer','dc_mw'])
run_model("A2: Time+Season+DC+Dummy2022", reg, 'ncent_mw', ['time','summer','dc_mw','post2022'])

# Test B: 2018-2024 half-year (14 points, original)
reg_24 = reg[reg['year'] <= 2024].copy()
reg_24['time'] = range(len(reg_24))
print("\n--- B: Half-year 2018-2024 ---")
run_model("B0: Time+Season", reg_24, 'ncent_mw', ['time','summer'])
run_model("B1: Time+Season+DC", reg_24, 'ncent_mw', ['time','summer','dc_mw'])

# Test C: Annual 2018-2025 (8 points)
annual = reg.groupby('year').agg(ncent_mw=('ncent_mw','mean'), dc_mw=('dc_mw','mean')).reset_index()
annual['time'] = range(len(annual))
annual['post2022'] = (annual['year'] >= 2022).astype(int)
print("\n--- C: Annual 2018-2025 ---")
run_model("C0: Time only", annual, 'ncent_mw', ['time'])
run_model("C1: Time+DC", annual, 'ncent_mw', ['time','dc_mw'])
run_model("C2: Time+DC+Dummy2022", annual, 'ncent_mw', ['time','dc_mw','post2022'])

# Test D: Annual 2018-2024 (7 points, original)
annual_24 = annual[annual['year'] <= 2024].copy()
annual_24['time'] = range(len(annual_24))
print("\n--- D: Annual 2018-2024 ---")
run_model("D0: Time only", annual_24, 'ncent_mw', ['time'])
run_model("D1: Time+DC", annual_24, 'ncent_mw', ['time','dc_mw'])

C:\Users\luozi\anaconda3\Lib\site-packages\dateutil\parser\_parser.py:1207: UnknownTimezoneWarning: tzname DST identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "
C:\Users\luozi\anaconda3\Lib\site-packages\dateutil\parser\_parser.py:1207: UnknownTimezoneWarning: tzname DST identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "
C:\Users\luozi\anaconda3\Lib\site-packages\dateutil\parser\_parser.py:1207: UnknownTimezoneWarning: tzname DST identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identifi

Total rows: 70128, Years: 2018-2025

Full dataset (16 rows):
 period  dc_mw  ncent_mw  year  summer  time  post2022
H1 2018  273.1  13415.43  2018       0     0         0
H2 2018  306.1  14320.11  2018       1     1         0
H1 2019  321.4  12814.68  2019       0     2         0
H2 2019  321.4  14806.80  2019       1     3         0
H1 2020  347.4  12467.45  2020       0     4         0
H2 2020  360.9  14059.87  2020       1     5         0
H1 2021  367.3  12779.58  2021       0     6         0
H2 2021  369.3  14051.63  2021       1     7         0
H1 2022  375.8  14083.27  2022       0     8         1
H2 2022  392.3  15291.94  2022       1     9         1
H1 2023  499.5  13378.21  2023       0    10         1
H2 2023  565.4  15761.72  2023       1    11         1
H1 2024  591.0  13923.37  2024       0    12         1
H2 2024  591.0  15541.81  2024       1    13         1
H1 2025  869.5  14667.37  2025       0    14         1
H2 2025 1067.3  15935.16  2025       1    15         1

CON

LinearRegression()

In [2]:
import warnings; warnings.filterwarnings('ignore')

In [3]:
# ============================================================
# 6. Test A2 + Population
# ============================================================
dfw_pop = pd.read_csv(os.path.join(DATA_DIR, "DFWPOP.csv"))
dfw_pop['year'] = pd.to_datetime(dfw_pop['observation_date']).dt.year
dfw_pop = dfw_pop[dfw_pop['year'].between(2017, 2025)][['year', 'DFWPOP']]

# Semi-annual interpolation (midpoint between adjacent July estimates)
pop_dict = dict(zip(dfw_pop['year'], dfw_pop['DFWPOP']))
pop_rows = []
for y in range(2018, 2026):
    prev = pop_dict.get(y - 1, pop_dict[y])
    curr = pop_dict[y]
    nxt = pop_dict.get(y + 1, curr + (curr - prev))  # extrapolate if missing
    pop_rows.append({'period': f'H1 {y}', 'pop': round((prev + curr) / 2, 3)})
    pop_rows.append({'period': f'H2 {y}', 'pop': round((curr + nxt) / 2, 3)})

pop_semi = pd.DataFrame(pop_rows)
reg2 = reg.merge(pop_semi, on='period')

print("--- A2 + Population ---")
run_model("A2+Pop: Time+Season+DC+Dummy+Pop", reg2, 'ncent_mw',
          ['time', 'summer', 'dc_mw', 'post2022', 'pop'])

# VIF check
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif_cols = ['time', 'summer', 'dc_mw', 'post2022', 'pop']
vif_data = reg2[vif_cols]
print("\nVIF check:")
for i, col in enumerate(vif_cols):
    print(f"  {col}: {variance_inflation_factor(vif_data.values, i):.1f}")

--- A2 + Population ---

A2+Pop: Time+Season+DC+Dummy+Pop
  n=16, R²=0.9365
  Coefficients: {'time': -244.951, 'summer': 1377.6485, 'dc_mw': 1.6142, 'post2022': 1308.949, 'pop': 2.3198}

VIF check:
  time: 34.2
  summer: 2.1
  dc_mw: 24.7
  post2022: 9.2
  pop: 8.3


In [4]:
# VIF for A2 without population
vif_cols_a2 = ['time', 'summer', 'dc_mw', 'post2022']
vif_data_a2 = reg[vif_cols_a2]
print("VIF check (A2 without pop):")
for i, col in enumerate(vif_cols_a2):
    print(f"  {col}: {variance_inflation_factor(vif_data_a2.values, i):.1f}")

VIF check (A2 without pop):
  time: 34.2
  summer: 1.9
  dc_mw: 16.8
  post2022: 9.1


In [5]:
import statsmodels.api as sm

# A2: Time + Season + DC + Dummy
X_a2 = sm.add_constant(reg[['time', 'summer', 'dc_mw', 'post2022']])
model_a2 = sm.OLS(reg['ncent_mw'], X_a2).fit()
print("=== A2: Time + Season + DC + Dummy ===")
print(model_a2.summary())

# A2+Pop: Time + Season + DC + Dummy + Population
X_a2p = sm.add_constant(reg2[['time', 'summer', 'dc_mw', 'post2022', 'pop']])
model_a2p = sm.OLS(reg2['ncent_mw'], X_a2p).fit()
print("\n=== A2+Pop: Time + Season + DC + Dummy + Pop ===")
print(model_a2p.summary())

=== A2: Time + Season + DC + Dummy ===
                            OLS Regression Results                            
Dep. Variable:               ncent_mw   R-squared:                       0.932
Model:                            OLS   Adj. R-squared:                  0.907
Method:                 Least Squares   F-statistic:                     37.66
Date:                Fri, 05 Jun 2026   Prob (F-statistic):           2.33e-06
Time:                        21:18:20   Log-Likelihood:                -112.40
No. Observations:                  16   AIC:                             234.8
Df Residuals:                      11   BIC:                             238.7
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1

In [6]:
# Test: replace DC with Pop
X_pop_only = sm.add_constant(reg2[['time', 'summer', 'post2022', 'pop']])
model_pop = sm.OLS(reg2['ncent_mw'], X_pop_only).fit()
print(model_pop.summary())

                            OLS Regression Results                            
Dep. Variable:               ncent_mw   R-squared:                       0.919
Model:                            OLS   Adj. R-squared:                  0.889
Method:                 Least Squares   F-statistic:                     31.06
Date:                Fri, 05 Jun 2026   Prob (F-statistic):           6.15e-06
Time:                        21:21:25   Log-Likelihood:                -113.83
No. Observations:                  16   AIC:                             237.7
Df Residuals:                      11   BIC:                             241.5
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -2.367e+04   1.78e+04     -1.328      0.2